<a href="https://colab.research.google.com/github/m-zayed5722/Miscellaneous-Projects/blob/main/PromptBench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PromptBench: Prompt Benchmarking & Optimization (Colab)

Goal:
Evaluate multiple prompt templates on the same tasks and automatically select the best one.

Pipeline:
Task → Prompt Variants → Generate Outputs → Score → Rank → Select Best Prompt

Features:
- Multiple prompt strategies
- Offline scoring (no API key)
- Prompt leaderboard
- Task-level evaluation
- Extensible to LLM-based evaluation later

In [1]:
#!pip -q install pandas numpy rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.7 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import pandas as pd
from typing import List, Dict
from rapidfuzz import fuzz

In [3]:
TASKS = [
    {
        "id": "t1",
        "input": "Explain RAG in simple terms",
        "reference": "RAG combines retrieval of relevant documents with generation so answers are grounded in external knowledge."
    },
    {
        "id": "t2",
        "input": "Give 3 guardrails for Text-to-SQL",
        "reference": "Use SELECT-only queries, enforce table allowlists, and apply query limits."
    },
    {
        "id": "t3",
        "input": "Why is evaluation important in LLM systems?",
        "reference": "Evaluation helps measure correctness, reliability, and detect issues like hallucination."
    }
]

In [4]:
PROMPTS = {
    "zero_shot": "Answer the following question:\n{input}",

    "structured": """You are an expert assistant.
Provide a clear and structured answer.

Question:
{input}

Answer:""",

    "concise": """Answer in 2-3 concise sentences:
{input}""",

    "step_by_step": """Think step by step and then answer clearly.

Question:
{input}

Steps:
1.

Final Answer:""",

    "bullet_points": """Answer using bullet points:
{input}"""
}

In [5]:
def mock_llm(prompt_name: str, task_input: str) -> str:
    base = task_input.lower()

    if prompt_name == "zero_shot":
        return f"{task_input}. It is important in AI systems."

    elif prompt_name == "structured":
        return f"Answer:\n{task_input} involves key concepts. It is important for reliability and performance."

    elif prompt_name == "concise":
        return f"{task_input} improves system reliability and accuracy."

    elif prompt_name == "step_by_step":
        return f"Step 1: Understand the concept.\nStep 2: Explain clearly.\nFinal Answer: {task_input} ensures better outcomes."

    elif prompt_name == "bullet_points":
        return f"- {task_input}\n- Improves reliability\n- Reduces errors"

    return task_input